# HFP — Dürüstlük Kontrolü: Exp LR Taraması (ctx=384)

## Motivasyon
Eğitilebilirlik testinde `exp` ctx=384'te çöktü (3 seed'in 2'si plato kıramadı). Ancak o test tek bir learning rate (`lr=1e-3`) kullanıyordu.
Belki `exp` modeli uzun bağlamda daha düşük veya daha yüksek bir LR'ye ihtiyaç duyuyordur?

Bilimsel dürüstlük ve titizlik gereği, bu çöküşün LR duyarlılığından kaynaklanıp kaynaklanmadığını kontrol etmeliyiz.

## Deney Tasarımı
- Görev: Dense retention (P=8, ctx=384)
- Mod: `exp` (cubic'e gerek yok, o zaten 1e-3'te çalıştı)
- LR'ler: `{3e-4, 3e-3}` (Önceki 1e-3 testine ek olarak)
- Seed'ler: `{0, 1, 2}`
- Bütçe: 600 adım

In [ ]:
# --- KURULUM ---
import subprocess, sys, os
REPO = "/kaggle/working/HFP"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "https://github.com/kayra-hn/HFP.git", REPO], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers>=4.40"], check=True)
os.chdir(REPO)
sys.path.insert(0, REPO)
os.environ["HFP_CKPT_DIR"] = "/kaggle/working/ck"
os.environ["PYTHONPATH"] = REPO
os.makedirs("/kaggle/working/ck", exist_ok=True)
import torch
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print("Kurulum tamam!")

In [ ]:
# --- LR TARAMASI (GPU) ---
import torch, random, math, time, numpy as np
from hfp.models.configuration_hfp import HFPConfig
from hfp.models.modeling_hfp import HFPForCausalLM

DEV = "cuda" if torch.cuda.is_available() else "cpu"

# --- Sabitler ---
KLO, KHI, VLO, VHI, FHI = 100, 130, 130, 160, 100
WIN, P, ANS = 8, 8, 0
STEPS, BS = 600, 16
CTX = 384
SEEDS = [0, 1, 2]
LR_LIST = [3e-4, 3e-3]
RET = "exp"
FMAP = "dpfp"
LN_VOCAB = math.log(VHI - VLO)
PLATO_THRESHOLD = 3.0

def make_seq(ctx):
    toks = [random.randint(1, FHI - 1) for _ in range(ctx)]
    slots = random.sample(range(ctx // 2), 2 * P)
    random.shuffle(slots)
    keys = random.sample(range(KLO, KHI), P)
    lab = [-100] * ctx
    for i in range(P):
        a, b = slots[2 * i], slots[2 * i + 1]
        if a > b: a, b = b, a
        wp, qp = 2 * a, 2 * b
        v = random.randint(VLO, VHI - 1)
        toks[wp] = keys[i]; toks[wp + 1] = v
        toks[qp] = keys[i]; toks[qp + 1] = ANS
        lab[qp + 1] = v
    return toks, lab

def build():
    cfg = HFPConfig(
        vocab_size=VHI + 4, hidden_size=64, num_hidden_layers=2,
        num_attention_heads=2, intermediate_size=256, bulk_dim=32,
        short_len=8, max_position_embeddings=CTX + 8, local_window=WIN,
        decay_mode=RET, rec_block=32, write_rule="additive",
        key_feature_map=FMAP, ffn_type="standard"
    )
    return HFPForCausalLM(cfg).to(DEV)

def train_and_measure(lr, seed, budget=1500):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if DEV == "cuda": torch.cuda.manual_seed(seed)
    model = build()
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=STEPS)
    model.train()
    t0 = time.time()
    loss_curve = []
    nan_flag = False
    for step in range(1, STEPS + 1):
        if time.time() - t0 > budget:
            print(f"    BUTCE ASILDI step {step}"); break
        xs, ys = [], []
        for _ in range(BS):
            t, l = make_seq(CTX); xs.append(t); ys.append(l)
        out = model(torch.tensor(xs, device=DEV), labels=torch.tensor(ys, device=DEV))
        if not torch.isfinite(out.loss):
            print(f"    NaN! step {step}")
            nan_flag = True; break
        opt.zero_grad(set_to_none=True)
        out.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sch.step()
        loss_val = out.loss.item()
        loss_curve.append(loss_val)
        if step % 200 == 0:
            print(f"    step {step}/{STEPS}  loss={loss_val:.3f}  ({time.time()-t0:.0f}s)", flush=True)
    final_loss = loss_curve[-1] if loss_curve else float('inf')
    broke_plateau = final_loss < PLATO_THRESHOLD and not nan_flag
    elapsed = time.time() - t0
    return {
        'final_loss': round(final_loss, 3),
        'broke_plateau': broke_plateau,
        'nan': nan_flag,
        'time_s': round(elapsed, 1)
    }

all_results = {}
total_t0 = time.time()

for lr in LR_LIST:
    print(f"\n{'#'*60}")
    print(f"  LR = {lr}  (mod={RET}, ctx={CTX})")
    print(f"{'#'*60}")
    for seed in SEEDS:
        tag = f"{RET}|lr{lr}|s{seed}"
        print(f"\n  --- {tag} ---")
        r = train_and_measure(lr, seed)
        all_results[tag] = r
        status = "NaN!" if r['nan'] else ("KIRDI" if r['broke_plateau'] else "PLATO")
        print(f"  => loss={r['final_loss']}  [{status}]  ({r['time_s']}s)")

total_elapsed = time.time() - total_t0
print(f"\nToplam sure: {total_elapsed/60:.1f} dakika")

# ========== SONUC TABLOSU ==========
print(f"\n\n{'='*60}")
print(f"EXP LR TARAMASI SONUCLARI (ctx={CTX})")
print(f"{'='*60}")
print(f"{'LR':10s} {'seed':>5s} {'son loss':>10s} {'durum':>8s}")
print("-"*60)
for tag, r in sorted(all_results.items()):
    parts = tag.split('|')
    lr_s, seed_s = parts[1].replace('lr', ''), parts[2]
    status = "NaN!" if r['nan'] else ("KIRDI" if r['broke_plateau'] else "PLATO")
    print(f"{lr_s:10s} {seed_s:>5s} {r['final_loss']:10.3f} {status:>8s}")

print(f"\n\n{'='*60}")
print("Ozet: (Plato Kiran / Gecerli Seed)")
for lr in LR_LIST:
    broke = sum(1 for s in SEEDS if all_results.get(f"{RET}|lr{lr}|s{s}", {}).get('broke_plateau', False))
    nans = sum(1 for s in SEEDS if all_results.get(f"{RET}|lr{lr}|s{s}", {}).get('nan', False))
    valid = len(SEEDS) - nans
    print(f"  LR {lr} : {broke}/{valid} (NaN: {nans})")
print("="*60)


## Olası Sonuçlar ve Yorumlar
- **Exp başka bir LR'de öğreniyorsa (örn. `3e-3`'te 3/3 Plato Kırıldı):** O zaman önceki testteki fark bir eğitilebilirlik kapasitesi farkı değil, bir *LR duyarlılığı* (LR sensitivity) farkıdır. Cubic'in iyi yanı LR'ye daha az duyarlı olması olur.
- **Exp hiçbir LR'de tam öğrenemiyorsa:** O zaman ctx=384'teki eğitilebilirlik başarısızlığı *yapısal* bir durumdur ve Cubic'in uzun bağlam eğitilebilirlik avantajı (Hypothesis B) kesinleşir.